# PyHamilton Demo

This notebook is a beginner's introduction to [pyhamilton](https://github.com/dgretton/pyhamilton), an open-source Python package that controls Hamilton liquid handlers. We use it to run this procedure on a 96-well plate:

1. Add **700 uL of internal standard** (in acetonitrile) from the **center trough** to every well
2. Add **100 uL of patient sample** (dilute urine) from the **glass tubes** to every well
3. **Mix** each well 3 times

| What | Where |
|---|---|
| Internal standard | Center trough |
| Patient samples 1-96 | Glass tubes on 3 sample carriers (32 tubes each) |
| Everything is prepared in | 2 mL deep-well 96-well plate |

## How pyhamilton works

Hamilton robots are normally programmed in **VENUS**, Hamilton's own software. Its methods are written in HSL (Hamilton Standard Language). pyhamilton does not replace VENUS. It runs one small VENUS method of its own, and that method takes orders from Python:

```
 This notebook (Python)           pyhamilton                     VENUS Run Control (HxRun.exe)
 ----------------------           ----------                     -----------------------------
 aspirate(ham_int, ...)  ---->  turns the call into a  ---->     pyhamilton's HSL method picks up
                                JSON command on a local          the command (HTTP GET), runs it
                                web server (127.0.0.1:3221)      on the robot or simulator, and
 the call returns        <----  waits for the result   <----     posts the result back (HTTP POST)
```

So every pipetting line in this notebook becomes **one VENUS step**. You can watch each step happen in Run Control.

## How to use this notebook

- Do the one-time install in the [README](README.md) first.
- In the VENUS System Configuration Editor, turn on **Simulation** mode. The deck is then drawn on screen and no robot moves.
- Run the cells **from top to bottom**. Each section explains the code below it.

## Reference

pyhamilton has [online API docs](https://dgretton.github.io/pyhamilton-docs/). They are older than the current code and do not cover every function used here. Where they don't, the links in this notebook go to the docstrings in the pyhamilton source code, at the same version we install.

| Topic | Where to read |
|---|---|
| Install, prerequisites, tutorial video | [pyhamilton README](https://github.com/dgretton/pyhamilton#readme) |
| Connecting to the robot | [`HamiltonInterface` docs](https://dgretton.github.io/pyhamilton-docs/interface.html#pyhamilton.interface.HamiltonInterface), [`interface.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L501) |
| Layouts and labware | [`deckresource` docs](https://dgretton.github.io/pyhamilton-docs/deckresource.html), [`deckresource.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py) |
| Every command option and its default | [`defaultcmds` docs](https://dgretton.github.io/pyhamilton-docs/defaultcmds.html), [`defaultcmds.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/defaultcmds.py) |
| Liquid classes | [`liquid_classes.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_classes.py), [`liquid_class_db.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_class_db.py) |

## Imports

Everything we need comes from the top-level `pyhamilton` package:

| Name | What it is | Defined in |
|---|---|---|
| `HamiltonInterface` | The connection to VENUS. It starts Run Control and sends commands. | [`interface.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L501) |
| `INITIALIZE` | A **command template**: the name of a VENUS command plus default values for its options | [`defaultcmds.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/defaultcmds.py#L26) |
| `LayoutManager`, `layout_item` | Read the deck layout file and look up labware by name | [`deckresource.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py#L60) |
| `Tip96`, `Plate96`, `Reservoir60mL`, `EppiCarrier32` | Labware classes. Each one knows how many positions it has and what they are called. | [`deckresource.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py#L470) |
| `tip_pick_up`, `aspirate`, `dispense`, `tip_eject` | Pipetting with the 8 independent channels | [`liquid_handling_wrappers.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_handling_wrappers.py#L100) |
| `initialize` | A one-line shortcut for the `INITIALIZE` command. We write the long form below, so you can see how commands are sent. | [`liquid_handling_wrappers.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_handling_wrappers.py#L61) |
| `check_liquid_class_exists`, `copy_liquid_class`, `set_tip_type` | Look up and edit VENUS liquid classes | [`liquid_class_db.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_class_db.py#L169), [`liquid_classes.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_classes.py#L54) |

The pipetting functions are thin wrappers. For example, `aspirate(ham_int, ...)` just calls the method `ham_int.aspirate(...)`. You may see either style in other pyhamilton examples.

In [2]:
from pyhamilton import (
    HamiltonInterface,
    INITIALIZE,
    LayoutManager,
    layout_item,
    Tip96,
    Plate96,
    Reservoir60mL,
    EppiCarrier32,
    initialize,
    tip_pick_up,
    aspirate,
    dispense,
    tip_eject,
    check_liquid_class_exists,
    copy_liquid_class,
    set_tip_type,
)

## Load the deck layout

A **deck layout** (`.lay`) file is made in the VENUS Method Editor. It records each piece of labware on the deck (carriers, tip racks, troughs, plates), where it sits, and its **name**.

[`LayoutManager`](https://dgretton.github.io/pyhamilton-docs/deckresource.html#pyhamilton.deckresource.LayoutManager) reads the file so we can look up labware by name. It also **installs** the layout into pyhamilton's own VENUS method, because that is the method Run Control actually runs ([source](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py#L161)). If the file differs from the one installed now, it prints `BACKING UP AND INSTALLING NEW LAYFILE`. It saves a copy in pyhamilton's `LAY-BACKUP` folder and copies our layout over pyhamilton's `STAR_OEM_Test.lay`. If the layout hasn't changed, it prints nothing, as in the run below.

The `r` before the path makes it a *raw string*, so Python leaves the Windows backslashes (`\`) alone.

In [3]:
layout_file = r"C:\Program Files (x86)\HAMILTON\Methods\Demo\msacl-2026-demo.lay"

lmgr = LayoutManager(layout_file)

BACKING UP AND INSTALLING NEW LAYFILE


## Define labware

Each piece of labware has a name in the layout file. You can see these names in the VENUS deck layout editor.

`layout_item(lmgr, LabwareClass, 'name')` finds that name in the layout and returns a Python object for it ([source](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py#L727)). It raises an error if the name is not in the layout, or if you already used it. That catches typos before the robot moves.

The **class** tells pyhamilton how many positions the labware has and what VENUS calls each one. We pick positions by a number (**index**) starting at 0, and the class turns it into VENUS's **position ID**:

| Class | Positions | Index 0 &rarr; ID | Last index &rarr; ID | Used for |
|---|---|---|---|---|
| [`Tip96`](https://dgretton.github.io/pyhamilton-docs/deckresource.html#pyhamilton.deckresource.Tip96) | 96 | `'1'` | 95 &rarr; `'96'` | Tip racks |
| [`Plate96`](https://dgretton.github.io/pyhamilton-docs/deckresource.html#pyhamilton.deckresource.Plate96) | 96 | `'A1'` | 95 &rarr; `'H12'` | The deep-well plate |
| [`Reservoir60mL`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py#L635) | 8 | `'1'` | 7 &rarr; `'8'` | The trough |
| [`EppiCarrier32`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/deckresource.py#L677) | 32 | `'1'` | 31 &rarr; `'32'` | The sample tube carriers |

pyhamilton doesn't have a class for every Hamilton part. Any class works as long as its **number of positions and IDs match** the labware in VENUS:
- Our 120 mL trough has 8 positions, one for each channel, the same as `Reservoir60mL`.
- Our tube carriers have 32 positions, the same as `EppiCarrier32`.

In [4]:
# 1000 uL filter tips
is_tips = layout_item(lmgr, Tip96, 'htf_l_0001')      # Step 1: internal standard
sample_tips = layout_item(lmgr, Tip96, 'htf_l_0002')  # Step 2: samples (8 new tips per batch)

# Center trough (position 2 of the 3-trough carrier) holds the internal standard.
# pyhamilton has no 120 mL trough class. Reservoir60mL works fine here:
# it gives us 8 spots in one column, one for each channel.
trough = layout_item(lmgr, Reservoir60mL, 'rgt_cont_120ml_BC_A00_0002')

# 2 mL deep-well 96-well plate
plate = layout_item(lmgr, Plate96, 'Cos_96_DW_2mL_0001')

# 3 sample carriers with 32 glass tubes each.
# EppiCarrier32 is pyhamilton's class for any 32-position tube carrier.
rack1 = layout_item(lmgr, EppiCarrier32, 'SMP_CAR_32_13x100_A00_0001')  # samples 1-32
rack2 = layout_item(lmgr, EppiCarrier32, 'SMP_CAR_32_13x100_A00_0002')  # samples 33-64
rack3 = layout_item(lmgr, EppiCarrier32, 'SMP_CAR_32_13x100_A00_0003')  # samples 65-96
sample_racks = [rack1, rack2, rack3]

print('IS tips:     ', is_tips.layout_name())
print('Sample tips: ', sample_tips.layout_name())
print('Trough:      ', trough.layout_name())
print('Plate:       ', plate.layout_name())
for rack in sample_racks:
    print('Sample rack: ', rack.layout_name())

IS tips:      htf_l_0001
Sample tips:  htf_l_0002
Trough:       rgt_cont_120ml_BC_A00_0002
Plate:        Cos_96_DW_2mL_0001
Sample rack:  SMP_CAR_32_13x100_A00_0001
Sample rack:  SMP_CAR_32_13x100_A00_0002
Sample rack:  SMP_CAR_32_13x100_A00_0003


## Liquid settings

Next we choose the volumes and **liquid classes**.

A liquid class is a saved set of pipetting settings in VENUS: flow rates, air gaps, blowout volume, settling times, and corrections that make the volume accurate. Each one is tuned for one liquid and one tip type. VENUS ships with many, and the names describe them:

| Part of the name | Meaning |
|---|---|
| `HighVolume` | For 1000 uL tips |
| `Filter` | For tips **with a filter**. VENUS will not use a class with the wrong tip type. |
| `Water`, `Acetonitrile` | The liquid it was tuned for |
| `DispenseJet` | Dispense from above, in a quick jet. Good for empty wells. |
| `DispenseSurface` | Dispense with the tip at or in the liquid. Good for wells that already have liquid. |
| `Empty` | Dispense everything in the tip (as opposed to `Part`, which leaves some behind) |

For the mix in step 3, we use 400 uL, half of the 800 uL in each well. That mixes well without pulling in air.

In [5]:
is_volume = 700      # uL of internal standard per well
sample_volume = 100  # uL of patient sample per well

prewet_cycles = 3    # pre-mix the IS tips 3x before the first dispense
mix_cycles = 3       # mix each well 3x after adding the sample
mix_volume = 400     # uL; each well holds 800 uL after step 2

# A liquid class tells the robot how to pipette (speeds, air gaps, etc.)
# Internal standard is in acetonitrile. Jet dispense into the empty wells.
is_liquid_class = 'HighVolumeFilter_Acetonitrile_DispenseJet_Empty'

# Samples are dilute urine, so we pipette them like water.
# Surface dispense, because the tip goes into the 700 uL already in the well.
sample_liquid_class = 'HighVolumeFilter_Water_DispenseSurface_Empty'

## Positions

pyhamilton describes a place on the deck as a **position tuple**: `(labware, index)`. The robot has 8 channels, so each pipetting command takes a **list of up to 8 positions**, one for each channel. Volumes are a matching list of up to 8 numbers.

pyhamilton turns the list into the text VENUS expects, using the layout name and position ID ([source](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L840-L853)). For the IS tips below, VENUS receives `htf_l_0001, 1;htf_l_0001, 2;...;htf_l_0001, 8` and the channel pattern `1111111100000000` (channels 1-8 on, 9-16 off).

Indexes start at 0 and run **down each column**:

| index | 0 | 1 | ... | 7 | 8 | 9 | ... | 95 |
|---|---|---|---|---|---|---|---|---|
| well | A1 | B1 | ... | H1 | A2 | B2 | ... | H12 |

So one column of a plate or tip rack is 8 indexes in a row. Column *c* (counting from 1) starts at index `(c - 1) * 8`.

Samples are numbered 1 to 96 across the three carriers, and sample *n* goes into well *n*:

| sample | 1-32 | 33-64 | 65-96 |
|---|---|---|---|
| carrier | `rack1` | `rack2` | `rack3` |
| tube index | 0-31 | 0-31 | 0-31 |
| wells | A1-H4 | A5-H8 | A9-H12 |

We work in batches of 8. Each carrier holds exactly 4 batches, so a batch never spans two carriers.

The cell below builds the positions for step 1. The `readable` helper prints each position as `(layout name, position ID)`, which is what VENUS receives.

In [6]:
def readable(positions):
    """Show each (labware, index) as the (layout name, position ID) that VENUS sees."""
    return [(labware.layout_name(), labware.position_id(i)) for labware, i in positions]

# Column 1 of the IS tip rack
is_tip_positions = [(is_tips, i) for i in range(8)]
print('IS tip positions: ', readable(is_tip_positions))

# 8 spots in the trough, one for each channel
trough_positions = [(trough, i) for i in range(8)]
print('Trough positions: ', readable(trough_positions))

# One volume for each channel
is_volumes = [is_volume] * 8
print('IS volumes: ', is_volumes)

sample_volumes = [sample_volume] * 8
print('Sample volumes: ', sample_volumes)

IS tip positions:  [('htf_l_0001', '1'), ('htf_l_0001', '2'), ('htf_l_0001', '3'), ('htf_l_0001', '4'), ('htf_l_0001', '5'), ('htf_l_0001', '6'), ('htf_l_0001', '7'), ('htf_l_0001', '8')]
Trough positions:  [('rgt_cont_120ml_BC_A00_0002', '1'), ('rgt_cont_120ml_BC_A00_0002', '2'), ('rgt_cont_120ml_BC_A00_0002', '3'), ('rgt_cont_120ml_BC_A00_0002', '4'), ('rgt_cont_120ml_BC_A00_0002', '5'), ('rgt_cont_120ml_BC_A00_0002', '6'), ('rgt_cont_120ml_BC_A00_0002', '7'), ('rgt_cont_120ml_BC_A00_0002', '8')]
IS volumes:  [700, 700, 700, 700, 700, 700, 700, 700]
Sample volumes:  [100, 100, 100, 100, 100, 100, 100, 100]


## Connect to the robot

[`HamiltonInterface`](https://dgretton.github.io/pyhamilton-docs/interface.html#pyhamilton.interface.HamiltonInterface) is the connection to VENUS. `windowed=True` means "run VENUS in a visible Run Control window", so you can watch every step. `start()` then does this ([source](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L557)):

1. It sends a test `ping` and waits 5 seconds.
2. If pyhamilton's method is already running in Run Control, the ping is answered and it prints `Interface already open`.
3. If not, it prints `Opening HSL application` and opens Run Control with pyhamilton's method. **Press the green Play button in Run Control.** The method then waits for commands from Python.

Then we send our first command. [`send_command`](https://dgretton.github.io/pyhamilton-docs/interface.html#pyhamilton.interface.HamiltonInterface.send_command) puts it in the queue and returns an ID. [`wait_on_response`](https://dgretton.github.io/pyhamilton-docs/interface.html#pyhamilton.interface.HamiltonInterface.wait_on_response) waits until VENUS reports that the command is finished. `INITIALIZE` gets the robot ready (on a real robot, the channels move to their home positions). We allow it up to 300 seconds, so there is time to press Play. The default is 60.

> **Tips**
> - The pyhamilton README uses `with HamiltonInterface() as ham_int:`. This starts and stops the connection automatically. In a notebook, it is easier to call `start()` and `stop()` ourselves, so the connection stays open from cell to cell.
> - Don't confuse `windowed=True` with pyhamilton's `simulating=True`. With `simulating=True`, pyhamilton only logs the commands and never talks to VENUS. To dry-run with the deck drawn on screen, use `windowed=True` and turn on Simulation mode in VENUS.

In [7]:
ham_int = HamiltonInterface(windowed=True)
ham_int.start()
ham_int.wait_on_response(ham_int.send_command(INITIALIZE), timeout=300)
print('Robot initialized')

Starting a new server thread
Sending ping to check if interface is open
Opening HSL application
started the server thread
Robot initialized


This is Run Control during the run, with **Running (Simulator)** in the yellow box.
- **Left: the deck.** In Simulation mode, VENUS draws each piece of labware the first time a step uses it, so most of the deck is still empty here.
- **Right: the Trace View.** The lines near the top show pyhamilton's method receiving a command from Python (`JSON received from Server: {"command": "channelTipPickUp", ...}`). The highlighted line at the bottom shows VENUS starting the matching step, `1000ul Channel Tip Pick Up`.

![Run Control during the run](images/connect_run_control.png)

## Make an acetonitrile liquid class for filter tips

VENUS ships `HighVolumeAcetonitrilDispenseJet_Empty`, but it is set up for 1000 uL tips **without** a filter.
VENUS will not let us use it with our filter tips.

So we make our own class:

- [`check_liquid_class_exists`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_class_db.py#L169) looks up the name in the VENUS liquid database (`C:\Program Files (x86)\Hamilton\Config\ML_STARLiquids.mdb`). It reads the database file directly, which is why the README asks you to install the Microsoft Access Database Engine.
- [`copy_liquid_class`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_classes.py#L54) and [`set_tip_type`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/liquid_classes.py#L80) are commands that VENUS carries out, so they need the connection from the previous cell. They copy the class and change its tip type to 1000 uL **with** filter.

This changes the VENUS database on this PC permanently, but only the first time. After that, the new class already exists and the cell just says so.

> **Note on tip type numbers:** VENUS stores tip types as numbers. On our system, the built-in `HighVolumeFilter_...` classes use **5**, and the non-filter `HighVolume_...` classes use **4**, so we use 5. pyhamilton's [`TipType`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/resources/enums.py#L5-L15) list has these two the other way around (`uL_1000 = 5`, `uL_1000_with_filter = 4`). If you're not sure, check what a built-in class with the same tips uses.

In [8]:
if check_liquid_class_exists(is_liquid_class):
    print(is_liquid_class, 'already exists')
else:
    copy_liquid_class(ham_int, 'HighVolumeAcetonitrilDispenseJet_Empty', is_liquid_class)
    # 5 = 1000 uL filter tip, the same tip type the HighVolumeFilter_Water classes use
    set_tip_type(ham_int, is_liquid_class, 5)
    print('Created', is_liquid_class)

HighVolumeFilter_Acetonitrile_DispenseJet_Empty already exists


## Step 1: Add 700 uL of internal standard

We use the **same 8 tips** for all 12 columns. The tips never touch a sample, so they stay clean.

### 1a. Pick up tips

[`tip_pick_up`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L1007) picks up one tip on each channel, from the positions in the list. Here that is column 1 of the IS tip rack (tips 1-8). It waits until VENUS is done. If something goes wrong (for example, no tip is found), the VENUS error becomes a Python error and the cell stops.

In [9]:
tip_pick_up(ham_int, is_tip_positions)

### 1b. Fill all 12 columns

Each pass through the loop does an [`aspirate`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L886) from the trough and a [`dispense`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L948) into one plate column. The loop only changes which wells we use: column *c* is plate indexes `(c - 1) * 8` to `(c - 1) * 8 + 7`. We give the liquid class (`liquidClass=...`) for both steps.

**Pre-wetting.** Acetonitrile evaporates quickly. On the **first** aspirate only (column 1), we mix it in the tips 3 times, aspirating and dispensing 700 uL back into the trough. This fills the air in the tips with solvent vapor, so the volumes we add to the plate are accurate.

**Extra options.** Any keyword we add, such as `mixCycles=3`, goes straight to the VENUS command. All options and their defaults are listed in [`defaultcmds.py`](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/defaultcmds.py#L50-L100) (`channelAspirate` and `channelDispense`). A few to know:

| Option | Default | Meaning |
|---|---|---|
| `mixCycles`, `mixVolume` | 0, 0 | Mix in place: one cycle is 1 aspirate + 1 dispense. In an **aspirate**, the mixing happens **before** the liquid is taken up. In a **dispense**, it happens **after**. |
| `liquidHeight` | 1.0 | How far (mm) above the bottom of the container the tip goes |
| `capacitiveLLD` | 0 (off) | Liquid level detection. It is off by default, so the tips go to `liquidHeight`. |
| `dispenseMode` | 8 | 8 means "use the mode in the liquid class", for example jet or surface |

In [10]:
for column in range(12):  # 0 = column 1, ..., 11 = column 12
    first_well = column * 8

    plate_column = [(plate, first_well + i) for i in range(8)]

    print('Adding internal standard to column', column + 1)
    if column == 0:
        # Pre-mix the tips on the first aspirate only
        aspirate(ham_int, trough_positions, is_volumes, liquidClass=is_liquid_class,
                 mixCycles=prewet_cycles, mixVolume=is_volume)
    else:
        aspirate(ham_int, trough_positions, is_volumes, liquidClass=is_liquid_class)
    dispense(ham_int, plate_column, is_volumes, liquidClass=is_liquid_class)

Adding internal standard to column 1
Adding internal standard to column 2
Adding internal standard to column 3
Adding internal standard to column 4
Adding internal standard to column 5
Adding internal standard to column 6
Adding internal standard to column 7
Adding internal standard to column 8
Adding internal standard to column 9
Adding internal standard to column 10
Adding internal standard to column 11
Adding internal standard to column 12


In the simulator, column 3 is being filled. **Green** marks the positions used by the step that is running. Columns 1-2 are dark because they already hold internal standard. In the IS tip rack (top left), column 1 is dark: those 8 tips are on the channels.

![Adding internal standard](images/step1_adding_internal_standard.png)

### 1c. Throw the tips away

When `tip_eject` gets no positions, it sends the tips to the **default waste** on the deck ([source](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L1030)). To put tips back in the rack instead, give it a list of tip positions, as for `tip_pick_up`.

In [11]:
tip_eject(ham_int)

After `tip_eject`, all 96 wells (dark) hold 700 uL of internal standard. The sample carriers are not drawn yet, because no step has used them.

## Steps 2 and 3: Add 100 uL of sample, then mix

Every batch of 8 samples gets **new tips**, so samples never touch each other:

1. Pick up 8 new tips
2. Aspirate 100 uL from 8 sample tubes
3. Dispense into one plate column, then **mix 3 times** (step 3)
4. Throw the tips away

The mixing is part of the dispense step. In a dispense, `mixCycles` and `mixVolume` mix **after** dispensing. This liquid class dispenses at the surface, so the tip is down in the 700 uL of internal standard, and the mix happens in the full 800 uL.

The loop below runs 12 batches (samples 1-96 into columns 1-12). Everything comes from one number, `first`, the index of the first sample in the batch. For example, **batch 5**:

| Code | Value for batch 5 | Meaning |
|---|---|---|
| `first = batch * 8` | `4 * 8 = 32` | Samples 33-40, tips 33-40, wells 33-40 |
| `sample_racks[first // 32]` | `sample_racks[1]` = `rack2` | `//` divides and drops the remainder, so 32 // 32 = 1 |
| `first % 32` | `0` | `%` is the remainder: tubes 1-8 of `rack2` |
| `(sample_tips, first + i)` | indexes 32-39 | Tip IDs `'33'`-`'40'` (column 5 of the rack) |
| `(plate, first + i)` | indexes 32-39 | Wells A5-H5 |

In [12]:
for batch in range(12):  # 0 = batch 1, ..., 11 = batch 12
    first = batch * 8  # index of the first sample, tip and well in this batch

    rack = sample_racks[first // 32]  # 32 tubes per carrier
    first_tube = first % 32           # tube index on that carrier

    sample_tip_positions = [(sample_tips, first + i) for i in range(8)]

    sample_positions = [(rack, first_tube + i) for i in range(8)]

    plate_column = [(plate, first + i) for i in range(8)]

    print(f'Samples {first + 1}-{first + 8} -> column {batch + 1}')
    tip_pick_up(ham_int, sample_tip_positions)
    aspirate(ham_int, sample_positions, sample_volumes, liquidClass=sample_liquid_class)
    dispense(ham_int, plate_column, sample_volumes, liquidClass=sample_liquid_class,
             mixCycles=mix_cycles, mixVolume=mix_volume)
    tip_eject(ham_int)

Samples 1-8 -> column 1
Samples 9-16 -> column 2
Samples 17-24 -> column 3
Samples 25-32 -> column 4
Samples 33-40 -> column 5
Samples 41-48 -> column 6
Samples 49-56 -> column 7
Samples 57-64 -> column 8
Samples 65-72 -> column 9
Samples 73-80 -> column 10
Samples 81-88 -> column 11
Samples 89-96 -> column 12


Batch 5 (samples 33-40) in the simulator, during the aspirate. The **green** tubes are positions 1-8 on `SMP_CAR_32_13x100_A00_0002`. In this layout, carrier `_0001` (samples 1-32) is the right-hand carrier (track 47), and `_0002` is just to its left (track 46). The second tip rack has 5 dark columns: one column of tips for each batch so far.

![Aspirating samples](images/step2_aspirating_samples.png)

Next, the dispense for batch 5. **Green** marks column 5 (wells A5-H5). The samples go in here, and then each well is mixed 3 times (`mixCycles=3`, `mixVolume=400`).

![Dispensing and mixing samples](images/step2_mixing.png)

The finished run: all three sample carriers are drawn now (from left: `_0003`, `_0002`, `_0001`). Every column of the second tip rack has been used, and every well holds internal standard + sample.

![Finished run](images/finished_run.png)

## Disconnect

[`stop()`](https://dgretton.github.io/pyhamilton-docs/interface.html#pyhamilton.interface.HamiltonInterface.stop) sends an `end` command, which tells pyhamilton's method in Run Control to finish. It then shuts down the local web server that carried the commands ([source](https://github.com/dgretton/pyhamilton/blob/65938d05cc8f5a5a0d3186648add560439ab1cdd/pyhamilton/interface.py#L619)). Always run this cell when you are done. Otherwise the next `HamiltonInterface` may find the old connection still open.

In [13]:
ham_int.stop()

Sending end command
Stopping server thread
Force closing HTTP server
Server thread run() method completed
Joining server thread with timeout
Server thread exited successfully
